# 13 · B3 (QLoRA + OCR) so với B2-sub (QLoRA, không OCR) — phép thử chính của W8

**Vấn đề:** Bước 6 cho thấy chèn OCR **chỉ lúc suy luận** không cải thiện có ý nghĩa (B1 +0,55 điểm EM, p = 0,55; B2 +1,00, p = 0,24). Phân tích dev300 cho thấy model **có dùng** OCR nhưng dùng không phân biệt: có lúc sửa đúng, có lúc chép nguyên lỗi OCR. Câu hỏi chính: **nếu train cùng OCR, model có học được cách dùng OCR đúng hay không?**

**Thiết kế (luật ghi trong `docs/W08/README.md` trước khi chạy):**

| Run | Dữ liệu train | OCR khi train | OCR khi test | Siêu tham số |
|---|---|---|---|---|
| **B2-sub** | tập con cố định 8.000 câu / 2.670 ảnh (seed 42) | không | không | QLoRA r 16, 1 epoch, cấu hình chốt W07 |
| **B3** | **cùng** tập con | VietOCR · v1 | VietOCR · v1 | **y hệt** B2-sub |

B3 so với B2-sub **chỉ khác đúng yếu tố OCR**. "Cải thiện" khi EM tăng **và** McNemar exact p < 0,05 trên test2000. Độ nhạy: ổn định khi lợi ích thật ≳ 2,3 điểm EM (README bước 6).

**Thứ tự:** chạy thử B3 (2 step) → train B2-sub → eval B2-sub → train B3 → eval B3 → so sánh.

**Runtime: L4.** Tổng khoảng **7 giờ**. Có thể chia **2 phiên**: dừng sau mục 5 (B2-sub xong cả train lẫn eval), hôm sau Run all lại thì phần đã xong được bỏ qua. Checkpoint lưu mỗi 100 step (khoảng 33 phút), nên nếu Colab ngắt thì mất tối đa khoảng 30 phút.

## 1. Thiết lập môi trường

Giống notebook 04/08/12 (cùng pin `transformers==5.17.0`). Dừng hẳn nếu GPU không phải L4.

*Kỳ vọng: commit mới nhất (có notebook 13), `GPU: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')
assert os.path.exists('notebooks/13_qlora_ocr_b3_vs_b2sub_qwen25vl.ipynb'), 'Code cũ -> git pull chưa lấy được commit notebook 13'

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (ổn nếu zip ảnh đã nằm trên Drive)')

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU'
print('GPU:', GPU)
if 'L4' not in GPU:
    raise SystemExit('❌ Phải chạy trên L4 (cùng GPU với B1/B2). Runtime -> Change runtime type -> L4, rồi chạy lại cell này.')
print('✅ Thiết lập xong')

## 2. Tên run + kiểm tra trước khi tốn GPU

- Tập con train phải đúng **sha256** trong manifest bước 3 (8.000 câu).
- Cấu hình OCR đọc từ kết quả bước 5 (`ocr_selection.json`), **không gõ tay**.
- OCR cache VietOCR phải đủ **2.670 ảnh train** và **1.486 ảnh test2000**.
- Selftest của `train_qlora` và `run_baseline` (không cần GPU).

*Kỳ vọng: `sha256 tập con OK (8000 câu)`, `paddledet_vietocr | v1 | 600 | 0.0`, `train 2670/2670 · test 1486/1486`, hai dòng `SELFTEST OK`.*

In [ ]:
import json, hashlib
from src.data.ocr import default_cache_path, load_ocr_cache, images_of
from src.eval.run_baseline import prepare_images, DATASETS

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
DS = 'vitextvqa_official'
EXP_ROOT = f'{DRIVE_ROOT}/experiments'
ADP_ROOT = f'{DRIVE_ROOT}/adapters'
SUB = f'{DRIVE_ROOT}/data/{DS}/train_subset8000_byimage_seed42.json'
TEST = f"{DRIVE_ROOT}/{DATASETS[DS]['data_rel']}"
OUT_DIR = f'{EXP_ROOT}/W08_B3_vs_B2sub_test2000_L4_seed42'
SAVE_STEPS = 100        # checkpoint ~33 phút/lần (không ảnh hưởng chữ ký run)

man = json.load(open('experiments/W08_data_subsets/train_subset8000_byimage_seed42_manifest.json', encoding='utf-8'))
h = hashlib.sha256(open(SUB, 'rb').read()).hexdigest()
assert h == man['output_sha256'], 'Tập con train trên Drive khác manifest!'
print(f"sha256 tập con OK ({man['n_questions']} câu, {man['n_images']} ảnh)")

sel = json.load(open('experiments/W08_ocr_select_dev300_L4_seed42/ocr_selection.json', encoding='utf-8'))['decision']
BACKEND, TPL, MAXC, MINC = sel['backend'], sel['template'], sel['max_chars'], sel['min_conf']
print(BACKEND, '|', TPL, '|', MAXC, '|', MINC)
OCR_ARGS = f'--mode ocr --ocr_backend {BACKEND} --ocr_template {TPL} --ocr_max_chars {MAXC} --ocr_min_conf {MINC}'

TAIL = 'qwen25vl_vitextvqa_sub8k_ep1_bf16_L4_seed42'
RUN = {
    'smoke':     f'W08_smoke_B3_qlora_ocr_{TAIL}',
    'B2sub':     f'W08_B2sub_qlora_r16_{TAIL}',
    'B3':        f'W08_B3_qlora_ocr_vietocr_v1_r16_{TAIL}',
    'B2sub_eval': 'W08_B2sub_eval_qlora_r16_sub8k_ep1_vitextvqa_test2000_bf16_L4_seed42',
    'B3_eval':    'W08_B3_eval_qlora_ocr_vietocr_v1_r16_sub8k_ep1_vitextvqa_test2000_bf16_L4_seed42',
}
IMG = prepare_images(DS, DRIVE_ROOT)
c = load_ocr_cache(default_cache_path(DRIVE_ROOT, DS, BACKEND))
tr, te = set(images_of([SUB], IMG)), set(images_of([TEST], IMG))
print(f"train {len(tr & set(c['images']))}/{len(tr)} · test {len(te & set(c['images']))}/{len(te)}")
assert tr <= set(c['images']) and te <= set(c['images']), 'OCR cache thiếu ảnh -> chạy notebook 10 Phần B'
!python -m src.adaptation.train_qlora --selftest 2>&1 | tail -1
!python -m src.eval.run_baseline --selftest 2>&1 | tail -1

## 3. Chạy thử B3: 32 câu, 2 step (≈ 3–4 phút)

Đây là lần đầu train **cùng OCR** trên GPU thật. Chạy 2 step để chắc: prompt có dòng OCR, che nhãn đúng, không tràn VRAM. Run thử có tên riêng (`W08_smoke_…`), không dùng cho kết quả.

*Kỳ vọng: dòng `OCR: … 100.0% mẫu có chữ`, rồi `== XONG: 2 step | … | VRAM đỉnh … GB | loss …`, `PEFT-only OK`.*

In [ ]:
if os.path.exists(f"{EXP_ROOT}/{RUN['smoke']}/train_log.json"):
    print('⏭ smoke: đã có -> bỏ qua')
else:
    rn = RUN['smoke']          # biến đơn để IPython thay vào lệnh !python
    !python -u -m src.adaptation.train_qlora --dataset {DS} --data_file {SUB} --run_name {rn} --limit 32 --max_steps 2 {OCR_ARGS}

## 4. Train B2-sub: QLoRA 1 epoch, KHÔNG OCR (≈ 2,7 giờ)

Cùng lệnh với B2 ở W07. Khác duy nhất: `--data_file` là tập con 8.000 câu, nên 500 step thay vì 2.198.

*Kỳ vọng: `lịch: 500 step tổng (500 step/epoch), warmup 15 step`; cuối cùng `== XONG: 500 step | ~19–20 s/step | VRAM đỉnh ~7 GB`, `💾 adapter epoch 1`. Bị ngắt thì chạy lại cell, sẽ có dòng `↻ CHẠY TIẾP từ checkpoint-…`.*

In [ ]:
if os.path.exists(f"{EXP_ROOT}/{RUN['B2sub']}/train_log.json"):
    print('⏭ B2-sub train: đã có -> bỏ qua')
else:
    rn = RUN['B2sub']
    !python -u -m src.adaptation.train_qlora --dataset {DS} --data_file {SUB} --run_name {rn} --epochs 1 --save_steps {SAVE_STEPS}

## 5. Eval B2-sub trên test2000 (≈ 35 phút) — điểm dừng an toàn nếu chia 2 phiên

Gắn adapter `epoch_1` của B2-sub, prompt zero-shot (không OCR), giống hệt cách eval B2 ở W07.

*Kỳ vọng: 4 lô, `== KẾT QUẢ: EM=…`. Muốn chia 2 phiên thì dừng ở đây: **Runtime → Disconnect and delete runtime**; hôm sau mở lại notebook, Run all, phần đã xong sẽ được bỏ qua.*

In [ ]:
if os.path.exists(f"{EXP_ROOT}/{RUN['B2sub_eval']}/metrics.json"):
    print('⏭ B2-sub eval: đã có -> bỏ qua')
else:
    AD = f"{ADP_ROOT}/{RUN['B2sub']}/epoch_1"
    rn = RUN['B2sub_eval']
    !python -u -m src.eval.run_baseline --dataset {DS} --run_name {rn} --adapter {AD}

## 6. Train B3: QLoRA 1 epoch CÓ OCR (≈ 3 giờ)

Y hệt mục 4, chỉ thêm dòng OCR vào prompt khi train (cấu hình đã đóng băng ở bước 5). Prompt dài hơn khoảng 140 ký tự, nên chậm hơn B2-sub một chút.

*Kỳ vọng: dòng `OCR: paddledet_vietocr.json … 100.0% mẫu có chữ`, `lịch: 500 step tổng`, cuối cùng `== XONG: 500 step`, `💾 adapter epoch 1`.*

In [ ]:
if os.path.exists(f"{EXP_ROOT}/{RUN['B3']}/train_log.json"):
    print('⏭ B3 train: đã có -> bỏ qua')
else:
    rn = RUN['B3']
    !python -u -m src.adaptation.train_qlora --dataset {DS} --data_file {SUB} --run_name {rn} --epochs 1 --save_steps {SAVE_STEPS} {OCR_ARGS}

## 7. Eval B3 trên test2000 (≈ 35 phút)

Gắn adapter `epoch_1` của B3, prompt **có** dòng OCR (cùng cấu hình như lúc train).

*Kỳ vọng: dòng `OCR: … 99.85% mẫu có chữ`, 4 lô, `== KẾT QUẢ: EM=…`.*

In [ ]:
if os.path.exists(f"{EXP_ROOT}/{RUN['B3_eval']}/metrics.json"):
    print('⏭ B3 eval: đã có -> bỏ qua')
else:
    AD = f"{ADP_ROOT}/{RUN['B3']}/epoch_1"
    rn = RUN['B3_eval']
    !python -u -m src.eval.run_baseline --dataset {DS} --run_name {rn} --adapter {AD} {OCR_ARGS}

## 8. Kết quả theo luật

- **Phép thử chính:** B2-sub vs B3, ghép cặp trên test2000. "Cải thiện" khi EM tăng **và** p < 0,05.
- **Chỉ báo cáo:** B2 (full train, W07) vs B3 và B2 vs B2-sub, để xem lượng dữ liệu train ảnh hưởng ra sao.
- Chi phí train (giờ, s/step, VRAM) của B2-sub và B3.

*Kỳ vọng: bảng điểm 4 dòng, bảng chi phí 2 dòng, dòng `➜ PHÉP THỬ CHÍNH …`, file `b3_vs_b2sub_summary.json`.*

In [ ]:
import pandas as pd
os.makedirs(OUT_DIR, exist_ok=True)
B2_FULL = 'W07_B2_eval_qlora_r16_ep1_vitextvqa_test2000_bf16_L4_seed42'
B1 = 'W07_B1_zeroshot_qwen25vl_vitextvqa_test2000_bf16_L4_seed42'
EVALS = {'B1 zero-shot': B1, 'B2 (full 35.159 câu, không OCR)': B2_FULL,
         'B2-sub (8.000 câu, không OCR)': RUN['B2sub_eval'], 'B3 (8.000 câu, có OCR)': RUN['B3_eval']}
rows = []
for k, r in EVALS.items():
    m = json.load(open(f'{EXP_ROOT}/{r}/metrics.json', encoding='utf-8'))
    rows.append({'run': k, 'EM': round(m['metrics']['exact_match'], 4), 'ANLS': round(m['metrics']['anls'], 4),
                 'token-F1 (phụ)': round(m['metrics'].get('token_f1', float('nan')), 4), 's/câu': m['meta']['sec_per_sample']})
display(pd.DataFrame(rows))

cost = []
for k in ('B2sub', 'B3'):
    meta = json.load(open(f"{EXP_ROOT}/{RUN[k]}/train_log.json", encoding='utf-8'))['meta']
    cost.append({'run': k, 'mẫu train': meta['num_samples'], 'step': meta['cost']['global_step'],
                 'giờ train': round(meta['cost']['train_runtime_sec'] / 3600, 2), 's/step': meta['cost']['sec_per_step'],
                 'VRAM đỉnh (GB)': meta['cost']['peak_vram_gb'], 'loss cuối': meta['cost']['final_loss'],
                 'tham số train (%)': meta['trainable_pct'], 'chạy tiếp từ': meta['resumed_from']})
display(pd.DataFrame(cost))

summary = {'what': 'B3 (QLoRA + OCR) vs B2-sub (QLoRA, không OCR), cùng tập con 8.000 câu, test2000 (W08)',
           'rule': 'B3 cải thiện nếu EM tăng và McNemar exact p < 0,05; ANLS kèm CI95 bootstrap ghép cặp (10.000, seed 42)',
           'ocr_config': sel, 'train_subset_sha256': man['output_sha256'], 'table': rows, 'train_cost': cost, 'paired': {}}
pairs = [('B2sub_vs_B3', RUN['B2sub_eval'], RUN['B3_eval'], 'B2sub', 'B3'),
         ('B2_vs_B3_info', B2_FULL, RUN['B3_eval'], 'B2', 'B3'),
         ('B2_vs_B2sub_info', B2_FULL, RUN['B2sub_eval'], 'B2', 'B2sub')]
for name, ra, rb, la, lb in pairs:
    out = f'{OUT_DIR}/paired_{name}.json'
    a_dir, b_dir = f'{EXP_ROOT}/{ra}', f'{EXP_ROOT}/{rb}'
    !python src/eval/compare_runs.py --run_a {a_dir} --run_b {b_dir} --label_a {la} --label_b {lb} --out {out} > /dev/null
    p = json.load(open(out, encoding='utf-8')); p.pop('only_b_question_ids', None)
    summary['paired'][name] = p
    print(f"{name:18s} EM {p['em_a']:.4f} -> {p['em_b']:.4f} | chỉ {la} đúng {p['n_only_a']}, chỉ {lb} đúng {p['n_only_b']}, "
          f"p={p['p_value']:.3g} | ΔANLS {p['anls_diff']:+.4f} CI95 [{p['anls_ci95'][0]:+.4f}; {p['anls_ci95'][1]:+.4f}]")
p = summary['paired']['B2sub_vs_B3']
v = ('CẢI THIỆN' if p['em_b'] > p['em_a'] and p['p_value'] < 0.05 else
     'KÉM HƠN có ý nghĩa' if p['em_b'] < p['em_a'] and p['p_value'] < 0.05 else 'KHÔNG KHÁC BIỆT có ý nghĩa')
summary['verdict_B3_vs_B2sub'] = v
print(f"\n➜ PHÉP THỬ CHÍNH — B3 so với B2-sub: {v} (EM {p['em_a']:.4f} -> {p['em_b']:.4f}, p={p['p_value']:.3g})")
json.dump(summary, open(f'{OUT_DIR}/b3_vs_b2sub_summary.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('->', f'{OUT_DIR}/b3_vs_b2sub_summary.json')

## 9. ⏸ DỪNG

Gửi Claude **output mục 8** (2 bảng + các dòng so sánh + dòng `➜ PHÉP THỬ CHÍNH`), kèm dòng `== XONG` của mục 4 và mục 6. Sau đó **Runtime → Disconnect and delete runtime**.